# Build training data

Loads past seasons, builds the feature table with every candidate feature and saves it to `data/`. Run once per offseason.

In [ ]:
import sys
sys.path.append('..')

from src import config, features

## 1. Build the feature table
One row per player per active game, with ids, the stats we predict and every candidate feature.

In [ ]:
table = features.build_feature_table(config.TRAIN_YEARS)
print(table.shape, 'seasons', table['season'].min(), '-', table['season'].max())
table.head()

## 2. Compare with the old CSVs (2014-2024)
Temporary check that the refactor computes the same numbers as the old notebooks. Once you're happy with it, delete this section and the old CSVs in `data/`.

Two differences are expected:
- **Stat corrections**: nflverse has fixed some stats since the old files were made, so a few games differ. A different game also shifts the season and last-n averages that include it.
- **More games per player**: the new data counts every active game (snap counts), including games with 0 targets. The old data only had games where the player touched the ball, so section 2.3 recomputes the player averages on the old set of games to compare like for like.

In [ ]:
import numpy as np
import pandas as pd

OLD_DIR = config.DATA_DIR
STATS = ['receptions', 'receiving_yards', 'receiving_tds', 'carries', 'rushing_yards', 'rushing_tds']


def compare(new, old, keys, new_col, old_col):
    """Share of rows where new and old values match, per stat. new_col/old_col are patterns like 'off_{}_last5'."""
    old = old.loc[:, ~old.columns.duplicated()]
    merged = new.merge(old, on=keys, suffixes=('', '__old'))
    result = {}
    for stat in STATS:
        n_col, o_col = new_col.format(stat), old_col.format(stat)
        if o_col not in old.columns:
            continue
        a = merged[n_col]
        b = merged[o_col + '__old'] if o_col + '__old' in merged else merged[o_col]
        same = np.isclose(a, b) | (a.isna() & b.isna())
        result[stat] = {'rows compared': len(merged), 'match %': round(100 * same.mean(), 2),
                        'max abs diff': round(np.nanmax(np.abs(a - b)), 2)}
    return pd.DataFrame(result).T


compare_years = table[table['season'] <= 2024]

### 2.1 Team totals per game
The raw sums before any averaging. Differences here are stat corrections only.

In [ ]:
offense_totals = features.team_totals(compare_years, 'ball_team')
defense_totals = features.team_totals(compare_years, 'opponent_team')

print('Offense')
display(compare(offense_totals, pd.read_csv(OLD_DIR / 'team_offense_2014_24.csv'), ['game_id', 'ball_team'], '{}', '{}'))
print('Defense')
display(compare(defense_totals, pd.read_csv(OLD_DIR / 'team_defense_2014_24.csv'), ['game_id', 'opponent_team'], '{}', '{}'))

### 2.2 Team averages
The old files use a 5-game window, so the last-n averages are recomputed with `n=5` here.

In [ ]:
team5 = features.defense_features(
    features.offense_features(compare_years[features.ID_COLS + config.PLAYER_STATS], n=5), n=5)

offense = compare_years[['game_id', 'ball_team'] + [c for c in table.columns if c.startswith('off_')]].drop_duplicates()
defense = compare_years[['game_id', 'opponent_team'] + [c for c in table.columns if c.startswith('def_')]].drop_duplicates()
offense5 = team5[['game_id', 'ball_team'] + [c for c in team5.columns if c.startswith('off_') and c.endswith('_last5')]].drop_duplicates()
defense5 = team5[['game_id', 'opponent_team'] + [c for c in team5.columns if c.startswith('def_') and c.endswith('_last5')]].drop_duplicates()

checks = {
    'off season avg': compare(offense, pd.read_csv(OLD_DIR / 'team_offense_season_avg_2014-24.csv'),
                              ['game_id', 'ball_team'], 'off_{}_season_avg', '{}_season_avg'),
    'off last 5': compare(offense5, pd.read_csv(OLD_DIR / 'team_offense_last5_avg_2014_24.csv'),
                          ['game_id', 'ball_team'], 'off_{}_last5', '{}_last5'),
    'def season avg': compare(defense, pd.read_csv(OLD_DIR / 'team_defense_season_avg_2014-24.csv'),
                              ['game_id', 'opponent_team'], 'def_{}_season_avg', '{}_season_avg'),
    'def last 5': compare(defense5, pd.read_csv(OLD_DIR / 'team_defense_last5_avg_2014_24.csv'),
                          ['game_id', 'opponent_team'], 'def_{}_last5', '{}_last5'),
}
pd.concat({name: df['match %'] for name, df in checks.items()}, axis=1)

### 2.3 Player averages on the old set of games
Keeps only the player-games that exist in the old data, then recomputes the averages with the new functions. This checks the calculations themselves.

**The old `player_last5_avg` and `player_career_avg` CSVs are wrong.** `career_avg.ipynb` and `last5_avg.ipynb` sort the data (keeping the old row labels) and then assign a result that went through `.reset_index(drop=True)` (new labels 0, 1, 2...). pandas matches rows by label, so each row got another row's average; re-running that code reproduces the old CSVs 100%. So last 5 and career are compared against a correct recalculation from the old weekly data instead.

In [ ]:
old_weekly = pd.read_csv(OLD_DIR / 'player_weekly_data_2014_24.csv')
same_games = compare_years[features.ID_COLS + config.PLAYER_STATS].merge(old_weekly[features.KEYS], on=features.KEYS)
same_games = features.player_season_avg(same_games)
same_games = features.player_last_n_avg(same_games, n=5)
same_games = features.player_career_avg(same_games)

# Correct last 5 and career averages from the old weekly data (the old CSVs for these are misaligned)
reference = old_weekly.sort_values(features.KEYS).reset_index(drop=True)
by_player = reference.groupby('player_id')[STATS]
reference = reference.join(by_player.transform(lambda s: s.shift().rolling(5, min_periods=1).mean()).add_suffix('_last5'))
reference = reference.join(by_player.transform(lambda s: s.shift().expanding().mean()).add_suffix('_career'))

checks = {
    'season avg': compare(same_games, pd.read_csv(OLD_DIR / 'player_season_avg_2014-24.csv'),
                          features.KEYS, '{}_season_avg', '{}_season_avg'),
    'last 5': compare(same_games, reference, features.KEYS, '{}_last5', '{}_last5'),
    'career': compare(same_games, reference, features.KEYS, '{}_career', '{}_career'),
}
pd.concat({name: df['match %'] for name, df in checks.items()}, axis=1)

### 2.4 Effect of counting every active game
How much the player averages move once games with 0 targets are included (WRs only).

In [ ]:
wrs = compare_years[compare_years['position'] == 'WR']
old_wr_games = old_weekly[old_weekly['position'] == 'WR']
print(f"WR active games: {len(wrs):,} now vs {len(old_wr_games):,} in the old data")

old_season = pd.read_csv(OLD_DIR / 'player_season_avg_2014-24.csv')
both = wrs.merge(old_season[features.KEYS + ['receptions_season_avg', 'receiving_yards_season_avg']],
                 on=features.KEYS, suffixes=('', '_old'))
both[['receptions_season_avg', 'receptions_season_avg_old',
      'receiving_yards_season_avg', 'receiving_yards_season_avg_old']].mean().round(2)

## 3. Save

In [ ]:
config.DATA_DIR.mkdir(exist_ok=True)
table.to_parquet(config.TRAINING_DATA, index=False)
print('Saved', config.TRAINING_DATA)

for method in ['snap', 'yards']:
    rows = features.select_training_rows(table, method)
    print(f"Training rows ({method}): {len(rows):,} from {rows['player_id'].nunique()} players")